In [0]:
# this is to have intervention signals for the future agent

In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG = "health_and_habit_guardian"
GOLD_DAILY = f"{CATALOG}.gold.daily_health"
GOLD_RISKS = f"{CATALOG}.gold.risk_flags"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

daily = spark.table(GOLD_DAILY)

In [0]:
# Rule 1: cumulative sleep debt (7-day rolling)
sleep_debt = (
    daily.filter(F.col("rolling_7d_sleep_debt") >= 3.5)
    .withColumn("risk_type", F.lit("SLEEP_DEBT"))
    .withColumn(
        "severity",
        F.when(F.col("rolling_7d_sleep_debt") >= 7.0, F.lit("HIGH")).otherwise(F.lit("MEDIUM")),
    )
    .withColumn("score", F.col("rolling_7d_sleep_debt"))
    .withColumn(
        "reason",
        F.concat(
            F.lit("Sleep debt of "),
            F.round(F.col("rolling_7d_sleep_debt"), 1).cast("string"),
            F.lit("h accumulated over the last 7 days (project heuristic)"),
        ),
    )
    .withColumn("recommended_action", F.lit("Suggest an earlier wind-down and a consistent bedtime"))
    .select("user_id", "date", "risk_type", "severity", "score", "reason", "recommended_action")
)

In [0]:
# Rule 2: single bad night (respects sleep_missing via sleep_hours IS NOT NULL)
low_sleep = (
    daily.filter(F.col("sleep_hours").isNotNull() & (F.col("sleep_hours") < 6.0))
    .withColumn("risk_type", F.lit("LOW_SLEEP"))
    .withColumn(
        "severity",
        F.when(F.col("sleep_hours") < 5.0, F.lit("HIGH")).otherwise(F.lit("MEDIUM")),
    )
    .withColumn("score", F.col("sleep_hours"))
    .withColumn(
        "reason",
        F.concat(
            F.lit("Only "),
            F.round(F.col("sleep_hours"), 1).cast("string"),
            F.lit("h slept (project heuristic, target 8h)"),
        ),
    )
    .withColumn("recommended_action", F.lit("Suggest recovery sleep tonight; avoid late caffeine"))
    .select("user_id", "date", "risk_type", "severity", "score", "reason", "recommended_action")
)

In [0]:
# Rule 3: activity drop (recent 7d average vs 28d baseline)
activity_drop = (
    daily.filter(F.col("activity_drop_pct").isNotNull() & (F.col("activity_drop_pct") >= 0.30))
    .withColumn("risk_type", F.lit("ACTIVITY_DROP"))
    .withColumn(
        "severity",
        F.when(F.col("activity_drop_pct") >= 0.50, F.lit("HIGH")).otherwise(F.lit("MEDIUM")),
    )
    .withColumn("score", F.round(F.col("activity_drop_pct") * 100.0, 1))
    .withColumn(
        "reason",
        F.concat(
            F.lit("7-day step average down "),
            F.round(F.col("activity_drop_pct") * 100.0, 0).cast("string"),
            F.lit("% vs 28-day baseline (project heuristic)")),
    )
    .withColumn("recommended_action", F.lit("Suggest a short walk or light movement break"))
    .select("user_id", "date", "risk_type", "severity", "score", "reason", "recommended_action")
)

In [0]:
# Rule 4: elevated resting HR vs the user's own 7-day average.
elevated_hr = (
    daily.filter(
        F.col("resting_hr").isNotNull()
        & F.col("resting_hr_7d_avg").isNotNull()
        & ((F.col("resting_hr") - F.col("resting_hr_7d_avg")) >= 5.0)
    )
    .withColumn("risk_type", F.lit("ELEVATED_RESTING_HR"))
    .withColumn("severity", F.lit("MEDIUM"))
    .withColumn("score", F.col("resting_hr") - F.col("resting_hr_7d_avg"))
    .withColumn(
        "reason",
        F.concat(
            F.lit("Resting HR "),
            F.round(F.col("resting_hr"), 0).cast("string"),
            F.lit(" bpm is 5+ bpm above the 7-day average (project heuristic)")),
    )
    .withColumn("recommended_action", F.lit("Suggest rest and hydration; keep monitoring"))
    .select("user_id", "date", "risk_type", "severity", "score", "reason", "recommended_action")
)

In [0]:
# Rule 5: low composite recovery score.
low_recovery = (
    daily.filter(F.col("recovery_score").isNotNull() & (F.col("recovery_score") < 50.0))
    .withColumn("risk_type", F.lit("LOW_RECOVERY"))
    .withColumn(
        "severity",
        F.when(F.col("recovery_score") < 35.0, F.lit("HIGH")).otherwise(F.lit("MEDIUM")),
    )
    .withColumn("score", F.col("recovery_score"))
    .withColumn(
        "reason",
        F.concat(
            F.lit("Composite recovery score "),
            F.round(F.col("recovery_score"), 0).cast("string"),
            F.lit("/100 below 50 (project heuristic)"),
        ),
    )
    .withColumn("recommended_action", F.lit("Suggest prioritising sleep and light activity today"))
    .select("user_id", "date", "risk_type", "severity", "score", "reason", "recommended_action")
)

In [0]:
from functools import reduce

risks = reduce(
    lambda a, b: a.unionByName(b),
    [sleep_debt, low_sleep, activity_drop, elevated_hr, low_recovery],
).withColumn("_processed_at", F.current_timestamp())

final_cols = ["user_id", "date", "risk_type", "severity", "score", "reason", "recommended_action", "_processed_at"]
risks_final = risks.select(*final_cols)
risks_final.createOrReplaceTempView("risk_updates")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_RISKS}
USING DELTA
AS SELECT * FROM risk_updates WHERE 1 = 0
""")

cols = ", ".join(final_cols)
spark.sql(f"""
MERGE INTO {GOLD_RISKS} AS target
USING (
    SELECT {cols}
    FROM (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY user_id, date, risk_type
            ORDER BY _processed_at DESC NULLS LAST
        ) AS _rn
        FROM risk_updates
    )
    WHERE _rn = 1
) AS source
ON target.user_id = source.user_id AND target.date = source.date AND target.risk_type = source.risk_type
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

display(spark.sql(f"SELECT risk_type, severity, COUNT(*) AS n FROM {GOLD_RISKS} GROUP BY risk_type, severity ORDER BY risk_type, severity"))

risk_type,severity,n
SLEEP_DEBT,MEDIUM,18
